In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

np.random.seed(42)

#### Task 1: Chi-Square Goodness-of-Fit Test
Scenario: A mobile-game studio expects that players choose among four character classes in equal proportions (25 % each). After a recent update, they collected the following counts from 400 new players:

Class	Warrior	Mage	Rogue	Healer

Count	120	85	110	85

1.Store the observed counts in a NumPy array.

2.Compute the expected counts under the null hypothesis of equal preference.

3.Run st.chisquare(observed, f_exp=expected).

4.Report the χ² statistic, degrees of freedom, and p-value.

5.At α = 0.05, state whether you reject or fail to reject H₀.

6.In a Markdown cell, answer: "Which class (if any) appears over- or under-represented, and by how much?"

Under H₀: equal preference (25% each)

In [5]:
observed = np.array([120, 85, 110, 85])
expected = np.array([100, 100, 100, 100])

In [4]:
chi2_stat, p_value = st.chisquare(observed, f_exp=expected)

df = len(observed) - 1

print("Chi-square statistic:", chi2_stat)
print("Degrees of freedom:", df)
print("p-value:", p_value)

Chi-square statistic: 9.5
Degrees of freedom: 3
p-value: 0.023331360430831553


Decision (α = 0.05)

Since p-value < 0.05 → Reject H₀

Conclusion: Player choices are not equally distributed among the four classes.

| Class   | Observed | Expected | Difference              |
| ------- | -------- | -------- | ----------------------- |
| Warrior | 120      | 100      | +20 (over-represented)  |
| Mage    | 85       | 100      | -15 (under-represented) |
| Rogue   | 110      | 100      | +10 (over-represented)  |
| Healer  | 85       | 100      | -15 (under-represented) |

Conclusion:

Warrior is the most over-represented (+20)

Rogue is slightly over-represented (+10)

Mage and Healer are under-represented (-15 each)


#### Task 2: Chi-Square Test of Independence
Scenario: A marketing team wants to know whether subscription tier (Free, Basic, Premium) is associated with churn status (Churned, Retained). They collected data from 600 customers:

 Free	Basic	Premium

Churned	90	60	30

Retained	110	140	170

1.Create the contingency table as a 2 × 3 NumPy array or pandas DataFrame.

2.Run st.chi2_contingency(table) and unpack χ², p-value, degrees of freedom, and expected frequencies.

3.Display the expected-frequency table.

4.Report the χ² statistic, df, and p-value.

5.State your decision and a plain-language interpretation (e.g. "There is / is not evidence of an association between subscription tier and churn.").

In [6]:
table = np.array([
    [90, 60, 30],     # Churned
    [110, 140, 170]   # Retained
])

In [7]:
chi2, p, df, expected = st.chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", df)
print("p-value:", p)
print("Expected frequencies:\n", expected)

Chi-square statistic: 42.85714285714286
Degrees of freedom: 2
p-value: 4.939576018831198e-10
Expected frequencies:
 [[ 60.  60.  60.]
 [140. 140. 140.]]


Decision (α = 0.05)

Since p-value < 0.05 → Reject H₀

Interpretation (Markdown Answer)

Conclusion:
There is evidence of an association between subscription tier and churn status.

Explanation (simple logic):
If there were no relationship, each tier would have similar churn proportions.
But:

- Free users churn much more than expected (90 vs 60)

- Premium users churn much less than expected (30 vs 60)

- Basic is close to expected

This means churn behavior depends on subscription tier

Intuition (important for understanding)

H₀: Subscription tier and churn are independent (no connection)

Expected table assumes:

Same churn rate across all tiers

But real data shows:

- Free → high churn

- Premium → low churn

These differences are too large to be random → very small p-value

So we reject H₀ and conclude:
Tier affects churn

#### Task 3: Compute Cramér's V
Using the χ² statistic and the contingency table from Task 2:

1.Write a function:
def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.

    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
2.Compute Cramér's V for the Task 2 result.

3.Classify the effect size using Cohen's benchmarks for df* = min(rows, cols) − 1:
Small ≈ 0.10, Medium ≈ 0.30, Large ≈ 0.50 (for 2 × 3 tables, df* = 1)

4.In a Markdown cell, answer: "Is the relationship between tier and churn weak, moderate, or strong? What does this mean for the marketing team?"

In [9]:
def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.

    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
    return np.sqrt(chi2 / (n * min_dim))

In [10]:
n = 600
min_dim = 1

v = cramers_v(42.9, n, min_dim)
print("Cramér's V:", v)

Cramér's V: 0.26739483914241874


Effect size classification (Cohen benchmarks)

For df* = 1:

Small ≈ 0.10
Medium ≈ 0.30
Large ≈ 0.50

0.267 ≈ Medium (moderate effect, close to 0.30)

The relationship between subscription tier and churn is moderate.

Interpretation:
This means that subscription tier has a meaningful impact on whether customers churn, but it is not the only factor influencing churn behavior.

For the marketing team, this suggests that:

Subscription tier should be considered an important factor in retention strategies
Free users may need targeted interventions (e.g., incentives, upgrades)
Premium users show higher retention, indicating successful engagement

Conclusion:
The relationship is strong enough to act on, but other factors should also be analyzed for a complete strategy.